# XRF55 Bench — WavMambaHAR

**Model:** WavMambaHAR baseline — XRF55 benchmark  
**Dataset:** XRF55 — 11 activities, 4620 train / 1980 test  
**Split:** train = reps 1–14, test = reps 15–20. No val.

---

## Required attached datasets

| Dataset name | Role | Mount path |
|---|---|---|
| `xrf55-amp-dataset` | bench/raw + processed arrays + stats.json | `/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/` |

---

## Hyperparameters (plain protocol defaults)

| Parameter | Value |
|---|---|
| Epochs | 40 |
| LR | 1e-3 |
| Batch | 32 |
| Seed | 42 |

In [ ]:
# Cell 1 — Install mamba-ssm (required by WavMambaHAR BiMamba layers)
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
print('Install done')

In [ ]:
# Cell 2 — Clone latest code from GitHub
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/WaveConvMamba')
if not CODE_PATH.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/imhoangt/WaveConvMamba.git',
         str(CODE_PATH)],
        check=True
    )
else:
    print('Repo already cloned.')

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

from xrf55_bench.trainer import run
print('Import OK  : xrf55_bench.trainer.run')

In [ ]:
# Cell 3 — Configuration
from xrf55_bench.config import TrainCfg, TrainCfg_for_protocol

SEEDS      = [42]               # single-seed default
# SEEDS    = [4, 8, 17, 42]    # multi-seed
BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/xrf55_amp_raw')
# BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/xrf55_amp_processed')
OUTPUT_DIR = Path('/kaggle/working/outputs/runs/xrf55_bench/wavmamba')

print(f'Seeds      : {SEEDS}')
print(f'Bench dir  : {BENCH_DIR}')
print(f'Output dir : {OUTPUT_DIR}')

for fname in ['stats.json', 'y_train.npy', 'y_test.npy', 'wavmamba/X_train.npy', 'wavmamba/X_test.npy']:
    p = BENCH_DIR / fname
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p}')

In [ ]:
# Cell 4 � Run training

# Protocol presets (uncomment one):
# cfg = TrainCfg_for_protocol('plain',    seeds=tuple(SEEDS))  # AdamW lr=1e-3, no sched,     40ep
# cfg = TrainCfg_for_protocol('xrf55',    seeds=tuple(SEEDS))  # Adam  lr=1e-3, MultiStepLR, 200ep
# cfg = TrainCfg_for_protocol('apwmamba', seeds=tuple(SEEDS))  # AdamW lr=4e-4, warmup+cosine, 120ep
cfg = TrainCfg_for_protocol('plain', seeds=tuple(SEEDS))
run(model_name='wavmamba', bench_dir=BENCH_DIR, output_dir=OUTPUT_DIR,
    train_cfg=cfg, num_workers=4)


In [ ]:
# Cell 5 — Results
import json

metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    s    = m['summary']
    seeds = m['config']['seeds']
    print(f"\n{'='*55}")
    print(f"  XRF55 Bench — WavMambaHAR")
    print(f"  Seeds       : {seeds}")
    print(f"  Accuracy    : {s['test_accuracy_mean']*100:.2f}% ± {s['test_accuracy_std']*100:.2f}%")
    print(f"  F1 Macro    : {s['test_f1_macro_mean']*100:.2f}% ± {s['test_f1_macro_std']*100:.2f}%")
    print(f"  Best epochs : {s['best_epochs']}")
    print(f"  Params      : {s['params_M']}M  |  Size: {s['model_size_mb']}MB")
    if s.get('macs_G'):
        print(f"  MACs        : {s['macs_G']:.3f}G")
    if s.get('latency_mean_ms') is not None:
        print(f"  Latency     : {s['latency_mean_ms']:.2f} ± {s['latency_std_ms']:.2f} ms")
    print(f"  Time        : {s['total_time_s']}s")
    print(f"{'='*55}")
    if len(seeds) == 1:
        ps = m['per_seed'].get(str(seeds[0]), {})
        if ps.get('test_f1_per_class'):
            class_names = [
                'Waving', 'Clap Hands', 'Fall on the Floor', 'Jumping', 'Running',
                'Sitting Down', 'Standing Up', 'Turning', 'Walking',
                'Stretch Oneself', 'Pat on Shoulder',
            ]
            print(f"\n  Per-class F1:")
            for i, v in enumerate(ps['test_f1_per_class']):
                print(f"    {class_names[i]:<20}: {v*100:.2f}%")
else:
    print('metrics.json not found — training may not have completed.')

In [ ]:
# Cell 6 — Plots and results
from IPython.display import Image, display

for fname in ['training_curve.png', 'confusion_matrix.png', 'seed_comparison.png']:
    p = OUTPUT_DIR / 'plots' / fname
    if p.exists():
        display(Image(str(p)))

zip_path = OUTPUT_DIR / 'results_summary.zip'
if zip_path.exists():
    print(f'ZIP ready  : {zip_path}')
else:
    print('ZIP not found — run Cell 4 first.')